# Tổng hợp perturbation XAI cho VG và PSG



In [ ]:
from pathlib import Path
import csv, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
REPO = DRIVE / 'graphml-project3'
import sys
sys.path[:0] = [str(REPO / 'src/mmxai/xai/xai-experiment'), str(REPO / 'src')]
from xai_experiment.paths import runs_dir
RUNS = runs_dir()
RESULTS_DIR = None  # hoặc đặt Path('/content/drive/MyDrive/.../xai-YYYYMMDD-HHMMSS')
if RESULTS_DIR is None:
    complete = sorted((p.parent for p in RUNS.glob('xai-*/summary.json')
                       if all((p.parent / name).is_file() for name in
                              ('xai_results.csv', 'selected_cases.json', 'explanations.json'))),
                      key=lambda p: p.stat().st_mtime)
    assert complete, f'Không tìm thấy lượt XAI hoàn tất trong {RUNS}'
    RESULTS_DIR = complete[-1]
RESULTS_DIR = Path(RESULTS_DIR)
ANALYSIS = RESULTS_DIR / 'analysis'
ANALYSIS.mkdir(exist_ok=True)
print('XAI:', RESULTS_DIR, '| report:', ANALYSIS)


In [ ]:
summary = json.loads((RESULTS_DIR / 'summary.json').read_text())
selected = json.loads((RESULTS_DIR / 'selected_cases.json').read_text())
explanations = json.loads((RESULTS_DIR / 'explanations.json').read_text())
manifest = json.loads(Path(summary['training_manifest']).read_text())
raw = pd.read_csv(RESULTS_DIR / 'xai_results.csv')
assert len(raw) == summary['n_rows'] == 216
assert len(selected) == summary['n_selected_cases'] == 24
assert len(explanations) == 216
case_index = {(c['dataset'], int(c['caption_id'])): int(c['candidate_row']) for c in selected}
assert len(case_index) == 24
SELECTION_SEED, MODEL_SEED_STEP, ROW_SEED_STEP = 42, 100000, 10

def recover_seeds(row):
    stored = int(row['seed'])
    if stored in (0, 1, 2):
        model_seed = stored
        random_seed = int(row['random_seed']) if 'random_seed' in row and pd.notna(row['random_seed']) else None
    else:
        j = case_index[row['dataset'], int(row['caption_id'])]
        offset = stored - SELECTION_SEED - j * ROW_SEED_STEP - int(row['requested_k'])
        assert offset % MODEL_SEED_STEP == 0, ('Không thể khôi phục model seed', row.to_dict())
        model_seed, random_seed = offset // MODEL_SEED_STEP, stored
    assert model_seed in (0, 1, 2)
    return pd.Series({'model_seed': model_seed, 'random_seed_fixed': random_seed})

fixed = raw.copy()
fixed['seed_original'] = fixed['seed']
fixed[['model_seed', 'random_seed_fixed']] = fixed.apply(recover_seeds, axis=1)
fixed['seed'] = fixed['model_seed'].astype(int)
fixed['random_seed'] = fixed['random_seed_fixed'].astype('Int64')
fixed = fixed.drop(columns=['random_seed_fixed'])
keys = ['dataset', 'caption_id', 'model_seed', 'requested_k']
assert not fixed.duplicated(keys).any()
assert fixed.groupby(['dataset','caption_id']).size().eq(9).all()
assert fixed.groupby(['dataset','model_seed','requested_k']).size().eq(12).all()
ex_keys = {(e['dataset'], int(e['caption_id']), int(e['seed']), int(e['requested_k']))
           for e in explanations}
assert ex_keys == set(map(tuple, fixed[keys].itertuples(index=False, name=None)))
for item in summary['summary']:
    subset = fixed[(fixed.dataset == item['dataset']) & (fixed.group == item['group'])
                   & (fixed.seed0_outcome == item['seed0_outcome'])
                   & (fixed.requested_k == item['k'])]
    assert len(subset) == item['n']
    assert math.isclose(subset.final_score_drop.mean(), item['mean_final_drop'], abs_tol=1e-8)
    assert math.isclose(subset.random_final_score_drop_mean.mean(), item['mean_random_drop'], abs_tol=1e-8)
    assert math.isclose(subset.fidelity_advantage.mean(), item['mean_advantage'], abs_tol=1e-8)
print('Đã đối chiếu', len(fixed), 'dòng,', len(explanations), 'explanation và', len(summary['summary']), 'nhóm summary.')
print('CSV gốc dùng random seed trong cột seed:', bool((raw['seed'] > 2).any()))


In [ ]:
fixed['top1_changed'] = fixed['rank_after'] > 1
fixed['gt_improved'] = fixed['gt_rank_after'] < fixed['gt_rank_before']
fixed['gt_worsened'] = fixed['gt_rank_after'] > fixed['gt_rank_before']
fixed['coverage_pct'] = 100 * fixed['screen_coverage']
fixed['sparsity_pct'] = 100 * fixed['sparsity']
fixed['has_path'] = fixed['n_explained_paths'] > 0
fixed['advantage_positive'] = fixed['fidelity_advantage'] > 0
aggregate = dict(n=('caption_id','size'), actual_k=('k','mean'),
                 final_drop=('final_score_drop','mean'), random_drop=('random_final_score_drop_mean','mean'),
                 advantage=('fidelity_advantage','mean'), positive_advantage=('advantage_positive','mean'),
                 top1_changed=('top1_changed','mean'), gt_improved=('gt_improved','mean'),
                 gt_worsened=('gt_worsened','mean'), coverage_pct=('coverage_pct','mean'),
                 sparsity_pct=('sparsity_pct','mean'), structural_validity=('structural_validity','mean'),
                 evidence_known=('evidence_known_rate','mean'), with_path=('has_path','mean'))
overall = fixed.groupby(['dataset','requested_k'], as_index=False).agg(**aggregate)
strata = fixed.groupby(['dataset','group','seed0_outcome','requested_k'], as_index=False).agg(**aggregate)
by_seed = fixed.groupby(['dataset','model_seed','requested_k'], as_index=False).agg(**aggregate)
training = pd.DataFrame([{'dataset': r['dataset'], 'model_seed': int(r['seed']),
                          'alpha': r['alpha'], 'test_full_MRR': r['test_full_MRR']}
                         for r in manifest['runs']]).sort_values(['dataset','model_seed'])
assert len(training) == 6
print('Checkpoint / retrieval gốc:'); display(training.round(3))
print('XAI tổng thể (mỗi dòng gồm 12 caption × 3 seed):'); display(overall.round(3))
print('Theo direct/indirect và outcome xác định ở seed 0:'); display(strata.round(3))
print('Theo model seed:'); display(by_seed.round(3))


In [ ]:
# Lấy mẫu lại caption, giữ ba seed của mỗi caption cùng nhau (12 cụm/dataset).
rng = np.random.default_rng(20261121)
ci_rows = []
for (dataset, k), subset in fixed.groupby(['dataset','requested_k']):
    case_means = subset.groupby('caption_id')['fidelity_advantage'].mean().to_numpy()
    assert len(case_means) == 12
    draws = rng.choice(case_means, size=(5000, len(case_means)), replace=True).mean(axis=1)
    lo, hi = np.quantile(draws, [0.025, 0.975])
    ci_rows.append({'dataset': dataset, 'k': int(k), 'n_captions': len(case_means),
                    'mean_advantage': float(case_means.mean()),
                    'bootstrap_low': float(lo), 'bootstrap_high': float(hi)})
advantage_ci = pd.DataFrame(ci_rows).sort_values(['dataset','k'])
print('Fidelity advantage, bootstrap 95% CI theo caption:'); display(advantage_ci.round(3))
print('CI mô tả 12 caption đã chọn, không suy rộng ra toàn bộ test.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, dataset in zip(axes, ('vg', 'psg')):
    part = overall[overall.dataset == dataset].sort_values('requested_k')
    xs = np.arange(len(part)); width = 0.36
    ax.bar(xs-width/2, part.final_drop, width, label='Cạnh XAI')
    ax.bar(xs+width/2, part.random_drop, width, label='Cạnh ngẫu nhiên cùng loại')
    ax.set_xticks(xs, part.requested_k.astype(str))
    ax.set_title(dataset.upper()); ax.set_xlabel('Số cạnh yêu cầu (k)')
    ax.set_ylabel('Giảm điểm fuse trung bình (z-score)')
    ax.grid(axis='y', alpha=0.25)
axes[0].legend(); fig.suptitle('Fidelity deletion trên tập case đã chọn')
fig.tight_layout(); fig.savefig(ANALYSIS / 'fidelity_vs_random.png', dpi=160)
display(fig); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, dataset in zip(axes, ('vg', 'psg')):
    part = strata[(strata.dataset == dataset) & (strata.requested_k == 3)].copy()
    labels = part['group'] + '\n' + part['seed0_outcome']
    ax.bar(labels, part.advantage)
    ax.set_title(dataset.upper()); ax.set_ylabel('XAI drop − random drop')
    ax.tick_params(axis='x', labelrotation=20)
    ax.grid(axis='y', alpha=0.25)
fig.suptitle('Fidelity advantage theo loại truy vấn, k=3')
fig.tight_layout(); fig.savefig(ANALYSIS / 'advantage_by_group.png', dpi=160)
display(fig); plt.close(fig)


In [ ]:
label_names = {}
for dataset in ('vg','psg'):
    path = DRIVE / f'mmxai_{dataset}_bundle/huy_extra_cn/labels.json'
    label_names[dataset] = json.loads(path.read_text()) if path.is_file() else []
ex_index = {(e['dataset'], int(e['caption_id']), int(e['seed']), int(e['requested_k'])):
            e['explanation'] for e in explanations}

def node_name(dataset, node):
    if node.startswith('label:') and label_names[dataset]:
        return label_names[dataset][int(node.split(':',1)[1])]
    return node

def edge_text(dataset, explanation):
    rows = []
    for edge in explanation['edges']:
        src = node_name(dataset, edge['source']); dst = node_name(dataset, edge['target'])
        rows.append(f"{src} -[{edge['relation']}]-> {dst}")
    return '; '.join(rows)

representatives = []
for (dataset, group, outcome), part in fixed[(fixed.model_seed == 0) & (fixed.requested_k == 3)].groupby(
        ['dataset','group','seed0_outcome']):
    midpoint = part.fidelity_advantage.median()
    row = part.iloc[(part.fidelity_advantage-midpoint).abs().argmin()]
    ex = ex_index[dataset, int(row.caption_id), 0, 3]
    representatives.append({'dataset': dataset, 'group': group, 'seed0_outcome': outcome,
        'caption_id': int(row.caption_id), 'query': row['query'],
        'top_image_id': row['image_id'], 'gt_image_id': row['ground_truth_image_id'],
        'gt_rank_before': int(row.gt_rank_before), 'gt_rank_after': int(row.gt_rank_after),
        'top_rank_after': int(row.rank_after), 'advantage': row.fidelity_advantage,
        'edges': edge_text(dataset, ex)})
case_table = pd.DataFrame(representatives).sort_values(['dataset','group','seed0_outcome'])
pd.set_option('display.max_colwidth', 140)
print('Case tiêu biểu (seed 0, k=3; gần median advantage của mỗi nhóm):')
display(case_table.round(3))


In [ ]:
fixed.to_csv(ANALYSIS / 'xai_results_corrected.csv', index=False)
for (dataset, seed), part in fixed.groupby(['dataset','model_seed']):
    (ANALYSIS / f'{dataset}_seed{seed}_corrected.json').write_text(
        part.to_json(orient='records', force_ascii=False, indent=2), encoding='utf-8')
overall.to_csv(ANALYSIS / 'overall.csv', index=False)
strata.to_csv(ANALYSIS / 'by_group_outcome.csv', index=False)
by_seed.to_csv(ANALYSIS / 'by_model_seed.csv', index=False)
advantage_ci.to_csv(ANALYSIS / 'advantage_bootstrap_ci.csv', index=False)
case_table.to_csv(ANALYSIS / 'representative_cases.csv', index=False)
training.to_csv(ANALYSIS / 'training_baseline.csv', index=False)

print('Đã lưu bảng CSV và hai biểu đồ tại:', ANALYSIS)
